In [3]:
from pathlib import Path

import torch 
from torch import nn
import torchvision
from torchvision import datasets, transforms

torch.__version__, torchvision.__version__

('2.11.0+cpu', '0.26.0+cpu')

## 1. Create Datasets and DataLoader

In [8]:
data_dir = Path('data/pizza_sushi_steak/')
data_dir

WindowsPath('data/pizza_sushi_steak')

In [9]:
train_dir = data_dir / 'train'
test_dir = data_dir / 'test'

train_dir, test_dir

(WindowsPath('data/pizza_sushi_steak/train'),
 WindowsPath('data/pizza_sushi_steak/test'))

In [14]:
train_transform = transforms.Compose([
    transforms.Resize(size=(64, 64)),
    transforms.TrivialAugmentWide(num_magnitude_bins=31),
    transforms.ToTensor()
])
test_transform = transforms.Compose([
    transforms.Resize(size=(64, 64)),
    transforms.ToTensor()
])

train_transform, test_transform

(Compose(
     Resize(size=(64, 64), interpolation=bilinear, max_size=None, antialias=True)
     TrivialAugmentWide(num_magnitude_bins=31, interpolation=InterpolationMode.NEAREST, fill=None)
     ToTensor()
 ),
 Compose(
     Resize(size=(64, 64), interpolation=bilinear, max_size=None, antialias=True)
     ToTensor()
 ))

In [15]:
train_dataset = datasets.ImageFolder(root=train_dir,
                                        transform=train_transform)

test_dataset = datasets.ImageFolder(root=test_dir,
                                        transform=test_transform)

train_dataset, test_dataset

(Dataset ImageFolder
     Number of datapoints: 225
     Root location: data\pizza_sushi_steak\train
     StandardTransform
 Transform: Compose(
                Resize(size=(64, 64), interpolation=bilinear, max_size=None, antialias=True)
                TrivialAugmentWide(num_magnitude_bins=31, interpolation=InterpolationMode.NEAREST, fill=None)
                ToTensor()
            ),
 Dataset ImageFolder
     Number of datapoints: 75
     Root location: data\pizza_sushi_steak\test
     StandardTransform
 Transform: Compose(
                Resize(size=(64, 64), interpolation=bilinear, max_size=None, antialias=True)
                ToTensor()
            ))

In [17]:
image, label = train_dataset[0]
image.shape, label

(torch.Size([3, 64, 64]), 0)

In [18]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(dataset=train_dataset,
                                    batch_size=32,
                                    shuffle=True)

test_dataloader = DataLoader(dataset=test_dataset,
                                    batch_size=32,
                                    shuffle=False)

train_dataloader, test_dataloader

(<torch.utils.data.dataloader.DataLoader at 0x20f558c9e50>,
 <torch.utils.data.dataloader.DataLoader at 0x20f5591f440>)

In [19]:
class_names = train_dataset.classes
class_names

['pizza', 'steak', 'sushi']

In [20]:
class_dict = train_dataset.class_to_idx
class_dict

{'pizza': 0, 'steak': 1, 'sushi': 2}

In [22]:
img, label = next(iter(train_dataloader))
img.shape, label.shape

(torch.Size([32, 3, 64, 64]), torch.Size([32]))

### 1.1 Create Datasets and DataLoader (Script mode)

In [31]:
# Create a directory going_modular scripts
import os 
os.mkdir('going_modular')

In [32]:
%%writefile going_modular/data_setup.py

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

def create_dataloader(
    train_dir: str,
    test_dir: str,
    transform: transforms.Compose,
    batch_size: int):
    """Create training and testing DataLoaders.

    Takes in a training directory and testing directory path and turns them into
    PyTorch Datasets and then intro PyTorch DataLoaders.
    """
    
    train_data = datasets.ImageFolder(train_dir, transform=transform)
    test_data = datasets.ImageFolder(test_dir, transform=transform)

    class_names = train_data.classes

    train_dataloader = DataLoader(train_data,
                                    batch_size=batch_size,
                                    shuffle=True,
                                    pin_memory=True)
    test_dataloader = DataLoader(test_data,
                                    batch_size=batch_size,
                                    shuffle=False, 
                                    pin_memory=True)

    return train_dataloader, test_dataloader, class_names


Writing going_modular/data_setup.py


In [38]:
data_transform = transforms.Compose([
    transforms.Resize(size=(64, 64)),
    transforms.ToTensor()
])

In [41]:
from going_modular import data_setup

train_dataloader, test_dataloader, class_names = data_setup.create_dataloader(train_dir,
                                                                                test_dir,
                                                                                transform=data_transform,
                                                                                batch_size=32)
train_dataloader, test_dataloader, class_names

(<torch.utils.data.dataloader.DataLoader at 0x20f559193d0>,
 ['pizza', 'steak', 'sushi'])

## 2. Making a model (TinyVGG)

In [42]:
class TinyVGG(nn.Module):
    def __init__(self, input_shape: int,
                        hidden_units: int,
                        output_shape: int):
        super().__init__()
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=input_shape,
                        out_channels=hidden_units,
                        kernel_size=3,
                        stride=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units,
                        out_channels=hidden_units,
                        kernel_size=3,
                        stride=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_units,
                        out_channels=hidden_units,
                        kernel_size=3,
                        stride=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units,
                        out_channels=hidden_units,
                        kernel_size=3,
                        stride=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.classifier_layer = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=hidden_units * 13 * 13,
                        out_features=output_shape)
        )
    
    def forward(self, x: torch.Tensor):
        # x = self.conv_block_1(x)
        # print(x.shape)
        # x = self.conv_block_2(x)
        # print(x.shape)
        # x = self.classifier_layer(x)
        # print(x.shape)
        # return x

        return self.classifier_layer(
            self.conv_block_2(self.conv_block_1(x))
            )

In [50]:
device = 'cuda' if torch.cuda.is_available() else "cpu"

model_0 = TinyVGG(input_shape=3,
                    hidden_units=10,
                    output_shape=len(class_names)).to(device)
model_0

TinyVGG(
  (conv_block_1): Sequential(
    (0): Conv2d(3, 10, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block_2): Sequential(
    (0): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier_layer): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=1690, out_features=3, bias=True)
  )
)

In [51]:
image_batch, label_batch = next(iter(train_dataloader))
image_batch.shape, label_batch.shape

c:\Users\mahdi\Desktop\programming\homework\env\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


(torch.Size([32, 3, 64, 64]), torch.Size([32]))

In [56]:
model_0.eval()
with torch.inference_mode():
    pred = model_0(image_batch.to(device))
pred.shape

torch.Size([32, 3])

### 2.1 Making a model (TinyVGG) as script mode

In [57]:
%%writefile going_modular/model.py
import torch
from torch import nn

class TinyVGG(nn.Module):
    def __init__(self, input_shape: int,
                        hidden_units: int,
                        output_shape: int):
        super().__init__()
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=input_shape,
                        out_channels=hidden_units,
                        kernel_size=3,
                        stride=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units,
                        out_channels=hidden_units,
                        kernel_size=3,
                        stride=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_units,
                        out_channels=hidden_units,
                        kernel_size=3,
                        stride=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units,
                        out_channels=hidden_units,
                        kernel_size=3,
                        stride=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.classifier_layer = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=hidden_units * 13 * 13,
                        out_features=output_shape)
        )
    
    def forward(self, x: torch.Tensor):
        # x = self.conv_block_1(x)
        # print(x.shape)
        # x = self.conv_block_2(x)
        # print(x.shape)
        # x = self.classifier_layer(x)
        # print(x.shape)
        # return x

        return self.classifier_layer(
            self.conv_block_2(self.conv_block_1(x))
            )

Overwriting going_modular/model.py


In [58]:
from going_modular import model

model_1 = model.TinyVGG(input_shape=3,
                        hidden_units=10,
                        output_shape=len(class_names)).to(device)

model_1

TinyVGG(
  (conv_block_1): Sequential(
    (0): Conv2d(3, 10, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block_2): Sequential(
    (0): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): Conv2d(10, 10, kernel_size=(3, 3), stride=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier_layer): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=1690, out_features=3, bias=True)
  )
)

In [60]:
model_1.eval()
with torch.inference_mode():
    pred = model_1(image_batch.to(device))
pred.shape

torch.Size([32, 3])

## 3. Create train_step() and test_step() and train()

In [61]:
# Creaet train_step()
def train_step(model: torch.nn.Module,
                dataloader: torch.utils.data.DataLoader,
                loss_fn: torch.nn.Module,
                optimizer: torch.optim.Optimizer,
                device=device):

    model.train()

    train_loss, train_acc = 0, 0

    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        y_pred = model(X)

        loss = loss_fn(y_pred, y)
        train_loss += loss.item()

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        # Calculate accuracy
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class==y).sum().item() / len(y_pred)
    
    # Adjust metrics to get AVG loss/accuracy
    train_loss /= len(dataloader)
    train_acc /= len(dataloader)
    return train_loss, train_acc

In [62]:
# Create test_step()
def test_step(model: torch.nn.Module,
                dataloader: torch.utils.data.DataLoader,
                loss_fn: torch.nn.Module,
                device=device):

    model.eval()

    test_loss, test_acc = 0, 0

    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)

            test_pred_logits = model(X)

            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()

            test_pred_labels = torch.argmax(torch.softmax(test_pred_logits, dim=1), dim=1)
            test_acc += (test_pred_labels == y).sum().item() / len(y)

    # Adjust metrics to get AVG loss/accuracy
    test_loss /= len(dataloader)
    test_acc /= len(dataloader)
    return test_loss, test_acc


In [63]:
from tqdm.auto import tqdm

def train(model: torch.nn.Module,
            train_dataloader: torch.utils.data.DataLoader,
            test_dataloader: torch.utils.data.DataLoader,
            optimizer: torch.optim.Optimizer,
            loss_fn: torch.nn.Module,
            epochs: int=5,
            device=device):

    results = {"train_loss": [],
                "train_acc": [],
                "test_loss": [],
                "test_acc": []}
    
    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model=model,
                                        dataloader=train_dataloader,
                                        loss_fn=loss_fn,
                                        optimizer=optimizer,
                                        device=device)
        
        test_loss, test_acc = test_step(model=model,
                                        dataloader=test_dataloader,
                                        loss_fn=loss_fn,
                                        device=device)
        print(f"Epoch: {epoch} | Train loss: {train_loss:.4f} | Train acc: {train_acc:.4f} | Test loss: {test_loss:.4f} | Test acc: {test_acc:.4f}")

        results['train_loss'].append(train_loss)
        results['train_acc'].append(train_acc)
        results['test_loss'].append(test_loss)
        results['test_acc'].append(test_acc)
        
    return results

c:\Users\mahdi\Desktop\programming\homework\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 3.1 Turn training functions into a script mode

In [67]:
%%writefile going_modular/engine.py

from tqdm.auto import tqdm

import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu' 
# Creaet train_step()
def train_step(model: torch.nn.Module,
                dataloader: torch.utils.data.DataLoader,
                loss_fn: torch.nn.Module,
                optimizer: torch.optim.Optimizer,
                device=device):

    model.train()

    train_loss, train_acc = 0, 0

    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        y_pred = model(X)

        loss = loss_fn(y_pred, y)
        train_loss += loss.item()

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        # Calculate accuracy
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class==y).sum().item() / len(y_pred)
    
    # Adjust metrics to get AVG loss/accuracy
    train_loss /= len(dataloader)
    train_acc /= len(dataloader)
    return train_loss, train_acc

# Create test_step()
def test_step(model: torch.nn.Module,
                dataloader: torch.utils.data.DataLoader,
                loss_fn: torch.nn.Module,
                device=device):

    model.eval()

    test_loss, test_acc = 0, 0

    with torch.inference_mode():
        for batch, (X, y) in enumerate(dataloader):
            X, y = X.to(device), y.to(device)

            test_pred_logits = model(X)

            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()

            test_pred_labels = torch.argmax(torch.softmax(test_pred_logits, dim=1), dim=1)
            test_acc += (test_pred_labels == y).sum().item() / len(y)

    # Adjust metrics to get AVG loss/accuracy
    test_loss /= len(dataloader)
    test_acc /= len(dataloader)
    return test_loss, test_acc

def train(model: torch.nn.Module,
            train_dataloader: torch.utils.data.DataLoader,
            test_dataloader: torch.utils.data.DataLoader,
            optimizer: torch.optim.Optimizer,
            loss_fn: torch.nn.Module,
            epochs: int=5,
            device=device):

    results = {"train_loss": [],
                "train_acc": [],
                "test_loss": [],
                "test_acc": []}
    
    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model=model,
                                        dataloader=train_dataloader,
                                        loss_fn=loss_fn,
                                        optimizer=optimizer,
                                        device=device)
        
        test_loss, test_acc = test_step(model=model,
                                        dataloader=test_dataloader,
                                        loss_fn=loss_fn,
                                        device=device)
        print(f"Epoch: {epoch} | Train loss: {train_loss:.4f} | Train acc: {train_acc:.4f} | Test loss: {test_loss:.4f} | Test acc: {test_acc:.4f}")

        results['train_loss'].append(train_loss)
        results['train_acc'].append(train_acc)
        results['test_loss'].append(test_loss)
        results['test_acc'].append(test_acc)
        
    return results


Overwriting going_modular/engine.py


In [69]:
from going_modular import engine

# engine.train()

## 4. Creating a function to save the model

In [80]:
from pathlib import Path
import torch

def save_model(model: torch.nn.Module,
                target_dir: str,
                model_name: str):

    target_dir_path = Path(target_dir)
    target_dir_path.mkdir(parents=True,
                            exist_ok=True)

    assert model_name.endswith('.pth') or model_name.endswith('.pt')
    model_save_path = target_dir_path / model_name

    print(f"[INFO] Saving model to: {model_save_path}")
    torch.save(obj=model.state_dict(), f=model_save_path)


### 4.1 Create a function to save model (script mode)

In [81]:
%%writefile going_modular/utils.py

from pathlib import Path
import torch

def save_model(model: torch.nn.Module,
                target_dir: str,
                model_name: str):

    target_dir_path = Path(target_dir)
    target_dir_path.mkdir(parents=True,
                            exist_ok=True)

    assert model_name.endswith('.pth') or model_name.endswith('.pt')
    model_save_path = target_dir_path / model_name

    print(f"[INFO] Saving model to: {model_save_path}")
    torch.save(obj=model.state_dict(), f=model_save_path)


Overwriting going_modular/utils.py


## 5 Train, evaluate and save the model

In [82]:
epochs = 5

model_0 = TinyVGG(input_shape=3,
                    hidden_units=10,
                    output_shape=len(class_names)).to(device)

# Setup loss function and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model_0.parameters(), lr=0.001)

# Start the timer
from timeit import default_timer as timer
start_time = timer()

# Train model_0
model_0_results = train(model=model_0,
                        train_dataloader=train_dataloader,
                        test_dataloader=test_dataloader,
                        optimizer=optimizer,
                        loss_fn=loss_fn,
                        epochs=epochs,
                        device=device)

# End the timer 
end_time = timer()
print(f"[INFO] Total training time: {end_time-start_time:.3f} seconds.")

save_model(model=model_0,
            target_dir='models',
            model_name='05_pytorch_going_modular_cell_mode.pth'
)

 20%|██        | 1/5 [00:00<00:03,  1.22it/s]

Epoch: 0 | Train loss: 1.1003 | Train acc: 0.2773 | Test loss: 1.0912 | Test acc: 0.5417


 40%|████      | 2/5 [00:01<00:02,  1.24it/s]

Epoch: 1 | Train loss: 1.0919 | Train acc: 0.4023 | Test loss: 1.0795 | Test acc: 0.5417


 60%|██████    | 3/5 [00:02<00:01,  1.26it/s]

Epoch: 2 | Train loss: 1.0876 | Train acc: 0.4023 | Test loss: 1.0595 | Test acc: 0.5417


 80%|████████  | 4/5 [00:03<00:00,  1.23it/s]

Epoch: 3 | Train loss: 1.0894 | Train acc: 0.2812 | Test loss: 1.0571 | Test acc: 0.5417


100%|██████████| 5/5 [00:04<00:00,  1.21it/s]

Epoch: 4 | Train loss: 1.0780 | Train acc: 0.4336 | Test loss: 1.0715 | Test acc: 0.4110
[INFO] Total training time: 4.119 seconds.
[INFO] Saving model to: models\05_pytorch_going_modular_cell_mode.pth


### 5.1 Train, evaluate and save the model (script mode)

In [87]:
%%writefile going_modular/train.py

import os 
from timeit import default_timer as timer

import torch
from torchvision import transforms
import data_setup, engine, model, utils

# Setup hyperprameters
EPOCHS = 5
BATCH_SIZE = 32
HIDDEN_UNITS = 10
LEARNING_RATE = 0.001

# Setup directories
train_dir = 'data/pizza_sushi_steak/train/'
test_dir = 'data/pizza_sushi_steak/test/'

# Setup device agnostic code
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Create transforms
data_transform = transforms.Compose([
    transforms.Resize(size=(64, 64)),
    transforms.ToTensor()
])

# Create DataLoader and get class_names
train_dataloader, test_dataloader, class_names = data_setup.create_dataloader(
                                train_dir=train_dir,
                                test_dir=test_dir,
                                transform=data_transform,
                                batch_size=BATCH_SIZE)

# Create model
model = model.TinyVGG(input_shape=3,
                        hidden_units=HIDDEN_UNITS,
                        output_shape=len(class_names)).to(device)

# Setup loss and optimizer
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(),
                                lr=LEARNING_RATE)

# Start the timer
start_time = timer()

# Start training
engine.train(model=model,
             train_dataloader=train_dataloader,
             test_dataloader=test_dataloader,
             loss_fn=loss_fn,
             optimizer=optimizer,
             epochs=EPOCHS,
             device=device)

# End the timer 
end_time = timer()
print(f"[INFO] Total training time: {end_time-start_time:.3f} seconds.")

# Save model
utils.save_model(model=model,
                 target_dir='models',
                 model_name='05_pytorch_going_modular_script_mode.pth')


Overwriting going_modular/train.py


In [88]:
!python going_modular/train.py

Epoch: 0 | Train loss: 1.0992 | Train acc: 0.4102 | Test loss: 1.0953 | Test acc: 0.4328
Epoch: 1 | Train loss: 1.0954 | Train acc: 0.3164 | Test loss: 1.0949 | Test acc: 0.2083
Epoch: 2 | Train loss: 1.0739 | Train acc: 0.3945 | Test loss: 1.0790 | Test acc: 0.3523
Epoch: 3 | Train loss: 1.0360 | Train acc: 0.5352 | Test loss: 1.0653 | Test acc: 0.3826
Epoch: 4 | Train loss: 1.0275 | Train acc: 0.4844 | Test loss: 1.0074 | Test acc: 0.4839
[INFO] Total training time: 5.019 seconds.
[INFO] Saving model to: models\05_pytorch_going_modular_script_mode.pth



  0%|          | 0/5 [00:00<?, ?it/s]c:\Users\mahdi\Desktop\programming\homework\env\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)

100%|██████████| 5/5 [00:05<00:00,  1.00s/it]
